In [6]:
import pandas as pd
import numpy as np
from cleaning.cleaner import *

In [7]:
df=pd.read_csv(r"C:\Users\Administrator\OneDrive\Desktop\Case_Study\car_rental.csv")
# df.info()
df.drop(columns=['Customer_ID_y'], inplace=True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7280 entries, 0 to 7279
Data columns (total 29 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Reservation_ID     7280 non-null   str    
 1   Customer_ID_x      7280 non-null   str    
 2   Vehicle_ID         7280 non-null   str    
 3   Vehicle_Class      7280 non-null   str    
 4   Booking_TS         7280 non-null   str    
 5   Pickup_TS          7280 non-null   str    
 6   Return_TS          7280 non-null   str    
 7   City               7280 non-null   str    
 8   Pickup_Lat         7280 non-null   float64
 9   Pickup_Lon         7280 non-null   float64
 10  Drop_Lat           7280 non-null   float64
 11  Drop_Lon           7280 non-null   float64
 12  Odo_Start          7280 non-null   str    
 13  Odo_End            7280 non-null   str    
 14  Fuel_Level         7280 non-null   str    
 15  Damage_Flag        7280 non-null   int64  
 16  GPS_Lat            7280 non-null   

1. Vehicle_ID trimming and canonical case.


In [8]:
df = clean_vehicle_id(df)
df['Vehicle_ID'].unique()

<StringArray>
['CAR-18', 'CAR-12', 'CAR-28', 'CAR-24', 'CAR-07', 'CAR-16', 'CAR-22',
 'CAR-20', 'CAR-02', 'CAR-25', 'CAR-03', 'CAR-26', 'CAR-06', 'CAR-05',
 'CAR-09', 'CAR-27', 'CAR-08', 'CAR-11', 'CAR-15', 'CAR-23', 'CAR-19',
 'CAR-13', 'CAR-01', 'CAR-10', 'CAR-04', 'CAR-30', 'CAR-17', 'CAR-29',
 'CAR-14', 'CAR-21']
Length: 30, dtype: str

2. Timestamp normalization; invalid minutes fix; duration must be positive.


In [9]:
df=clean_timestamps(df)


c:\Users\Administrator\OneDrive\Desktop\Case_Study\cleaning\cleaner.py:23: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(col, errors='coerce')


3. Odometer numeric extraction and unit unification (km).


In [10]:
df = prepare_odo(df)
df = clean_odo_start(df)
df = clean_odo_end(df)
df['Odo_Start_km'].isnull().value_counts()
df['Odo_End_km'].isnull().value_counts()

Odo_End_km
False    7280
Name: count, dtype: int64

4. Fuel level normalization (50%→0.5).


In [11]:
df=clean_fuel_level(df)
df['Fuel_Fraction'].unique()


array([0.93, 0.9 , 0.94, 0.07, 0.4 , 0.41, 0.26, 0.92, 0.23, 0.08, 0.28,
       0.66, 0.11, 0.13, 0.77, 0.72, 0.19, 0.62, 0.7 , 0.96, 0.87, 0.06,
       0.61, 0.5 , 0.68, 0.95, 0.6 , 0.99, 0.83, 0.33, 0.3 , 0.8 , 0.36,
       0.45, 0.52, 0.43, 0.63, 0.39, 0.31, 0.09, 0.21, 0.51, 0.75, 0.54,
       0.56, 0.1 , 0.32, 0.34, 0.27, 0.24, 0.78, 1.  , 0.91, 0.89, 0.65,
       0.18, 0.82, 0.79, 0.44, 0.58, 0.2 , 0.53, 0.05, 0.25, 0.81, 0.98,
       0.35, 0.12, 0.38, 0.97, 0.37, 0.17])

5. Rate parsing to numeric daily rate; currency normalization.


In [12]:
df=clean_rate(df)
df['Daily_Rate'].unique()

array([1500,  800, 2500, 1200, 2000, 1000, 3000])

6. City normalization to canonical names.


In [13]:
df['City'].unique()

<StringArray>
[  'Kolkata',     'delhi',   'Chennai',   'kolkata',    'Mumbai',   'CHENNAI',
 'Hyderabad', 'BENGALURU',    'mumbai',     'Delhi',       'blr', 'Bengaluru',
       'HYD', 'bangalore']
Length: 14, dtype: str

In [14]:
df['City'].unique()
df=clean_city(df)
df['City'].unique()

<StringArray>
['Kolkata', 'Delhi', 'Chennai', 'Mumbai', 'Hyderabad', 'Bengaluru']
Length: 6, dtype: str

7. Duplicate reservation dedup (same ID).


In [15]:
# count duplicate IDs
df['Reservation_ID'].duplicated().sum()

np.int64(280)

In [16]:
df=remove_duplicate_reservation(df)
df['Reservation_ID'].unique()

<StringArray>
['RES-01778', 'RES-00231', 'RES-05855', 'RES-02940', 'RES-04969', 'RES-02516',
 'RES-06446', 'RES-06643', 'RES-06346', 'RES-05450',
 ...
 'RES-01077', 'RES-04569', 'RES-01681', 'RES-02941', 'RES-03041', 'RES-03127',
 'RES-02101', 'RES-02963', 'RES-02001', 'RES-00585']
Length: 7000, dtype: str

8. Return before pickup rule validation.


In [17]:
# df['Pickup_TS'].unique()
df=validate_trip_time(df)
df['Pickup_TS'].isnull().value_counts()
df['Return_TS'].isnull().value_counts()

Return_TS
False    7000
Name: count, dtype: int64

9. Payment method standardization (UPI/CARD/CASH/WALLET).


In [18]:
df['Payment'].unique()

<StringArray>
['wallet', 'card', 'upi', 'UPI', 'cash', 'Card']
Length: 6, dtype: str

In [19]:
df=clean_payment(df)
df['Payment'].unique()

<StringArray>
['WALLET', 'CARD', 'UPI', 'CASH']
Length: 4, dtype: str

10. Mileage sanity check (End ≥ Start).


In [20]:
df=mileage_sanity_check(df)
df['mileage_flag'].unique()
df['mileage_flag'].value_counts()
# df['mileage_flag'].isnull().sum()

mileage_flag
HIGH     3328
VALID    3144
ERROR     528
Name: count, dtype: int64

11. Refueling event detection vs fuel change and distance.


In [21]:
df=fuel_sanity_check(df)
df['fuel_flag'].unique()

<StringArray>
['NORMAL', 'HIGH_CONSUMPTION']
Length: 2, dtype: str

12. Vehicle availability overlap checks.


In [22]:
df=trip_overlap_check(df)
df['overlap_flag'].unique()

<StringArray>
['NO_PREVIOUS', 'OVERLAP', 'NO_OVERLAP', 'NEW_VEHICLE']
Length: 4, dtype: str

13. Damage/incident log linkage to reservation.


In [23]:
damage_cases, damage_rate = damage_analysis(df)

print(damage_cases.head())
print(damage_rate)

    Reservation_ID Customer_ID_x Vehicle_ID Vehicle_Class          Booking_TS  \
176      RES-03927         C3249     CAR-01           SUV 2026-01-01 22:29:00   
244      RES-03288         C2726     CAR-01       Economy 2026-01-02 17:34:00   
456      RES-00345         C6119     CAR-01           SUV 2026-01-05 08:24:00   
504      RES-00147         C6152     CAR-01       Compact 2026-01-05 21:13:00   
566      RES-01978         C8739     CAR-01       Economy 2026-01-06 17:19:00   

              Pickup_TS           Return_TS       City  Pickup_Lat  \
176 2026-01-01 22:29:00 2026-01-06 14:29:00  Bengaluru   12.966111   
244 2026-01-02 17:34:00 2026-01-11 07:34:00    Kolkata   22.571409   
456 2026-01-05 08:24:00 2026-01-09 01:24:00    Kolkata   22.565932   
504 2026-01-05 21:13:00 2026-01-06 22:13:00    Kolkata   22.588572   
566 2026-01-06 17:19:00 2026-01-16 20:19:00     Mumbai   19.086741   

     Pickup_Lon  ...  GST_Amount  Total_Amount  Rate_Plan  \
176   77.589671  ...       150.

14. Driver license masking and validation (if present).


In [24]:
df = clean_driver_license(df)

print(df['Driver_License'].head())

31     DL***43
66     DL***43
143    DL***43
176    DL***71
188    UNKNOWN
Name: Driver_License, dtype: str


15. Promo/coupon code validation & expiry checks.


In [25]:
df=clean_promo_code(df)
print(df['Promo_Code'].head())


31      BOGUS99
66       DISC15
143      DISC15
176    NO_PROMO
188    NO_PROMO
Name: Promo_Code, dtype: str


16. Telematics GPS join and jitter smoothing.


In [26]:
df=smooth_gps(df)
df[["GPS_Lat", "GPS_Lat_Smoothed", "GPS_Lon", "GPS_Lon_Smoothed"]].head()
df[["GPS_Lat", "GPS_Lat_Smoothed", "GPS_Lon", "GPS_Lon_Smoothed"]].isnull().sum()

GPS_Lat             0
GPS_Lat_Smoothed    0
GPS_Lon             0
GPS_Lon_Smoothed    0
dtype: int64

17. Speeding/harsh events normalization from telematics.


In [27]:
df=normalize_harsh_events(df)
df['Harsh_Events'].unique()
df['Harsh_Events'].isnull().sum()

np.int64(0)

18. PII redaction in notes and feedback.


In [28]:
df=redact_pii(df)
df[['Notes', 'Customer_Feedback']].head()

,Notes,Customer_Feedback
31,Bad Seats Quality,Good speaker
66,Contact Rahul **********,Good mileage
143,Speakers not working,email *****@*****
176,Bad Seats Quality,email *****@*****
188,scratch on door,Good speaker


19. Rate plan mapping to master tariffs.


In [29]:
df=prepare_rate_plan_data(df)
df[['Rate_Plan', 'Daily_Rate']].isnull().sum()

Rate_Plan     0
Daily_Rate    0
dtype: int64

20. Tax/GST computation validation.

In [30]:
df=validate_gst(df)
df[['Daily_Rate','GST_Amount','Total_Amount']].head()
df[['Daily_Rate','GST_Amount','Total_Amount']].isnull().sum()

Daily_Rate      0
GST_Amount      0
Total_Amount    0
dtype: int64

In [31]:
# Max_Speed_kmh
# checking the unique values in the 'Max_Speed_kmh' column
df['Max_Speed_kmh'].unique()

# checking for null values in the 'Max_Speed_kmh' column
df['Max_Speed_kmh'].isnull().sum()

# removing 'km/h'
df['Max_Speed_kmh'] = df['Max_Speed_kmh'].str.replace('km/h', '')

# removing 'kmh'
df['Max_Speed_kmh'] = df['Max_Speed_kmh'].str.replace('kmh', '')

# removing leading and trailing spaces
df['Max_Speed_kmh'] = df['Max_Speed_kmh'].str.strip()

# converting to int
df['Max_Speed_kmh'] = df['Max_Speed_kmh'].astype(int)

# checking the unique values after cleaning
df['Max_Speed_kmh'].unique()

array([ 95,  81,  42, 163,  68,  89, 139,  46,  75, 124, 143,  78,  50,
       146,  54, 122,  44,  49, 142, 145, 177, 161, 164, 165,  92,  80,
       153,  97,  51, 129,  96,  66,  77,  56, 125,  71, 134, 120,  53,
       140, 169,  73, 137,  74, 130,  40,  60,  76, 131, 121, 166,  87,
        45, 148,  64,  94,  82,  88, 162,  70, 178, 160,  84,  83, 171,
       136, 126,  58, 176, 128, 159,  52, 179, 155, 170,  43, 154, 172,
       147, 123,  85, 100, 151, 175, 174,  93])

In [32]:
# columns to clean (latitude and longitude)
cols = ['GPS_Lat','GPS_Lon','Drop_Lat','Drop_Lon','Pickup_Lat','Pickup_Lon']

# cleaning latitude and longitude columns
for col in cols:

    # checking the unique values in the column
    df[col].unique()

    # checking for null values
    df[col].isnull().sum()

    # converting to string
    df[col] = df[col].astype(str)

    # removing unwanted characters (keeping digits, '.', '-')
    df[col] = df[col].apply(lambda x: re.sub(r'[^0-9\.\-]', '', x))

    # converting to numeric
    df[col] = pd.to_numeric(df[col], errors='coerce')

    # creating validation flag
    if 'Lat' in col:
        df[col + '_Flag'] = df[col].apply(
            lambda x: 'INVALID' if (pd.notnull(x) and (x < -90 or x > 90)) else 'VALID'
        )
    else:
        df[col + '_Flag'] = df[col].apply(
            lambda x: 'INVALID' if (pd.notnull(x) and (x < -180 or x > 180)) else 'VALID'
        )

    # setting invalid values to NaN
    if 'Lat' in col:
        df.loc[(df[col] < -90) | (df[col] > 90), col] = np.nan
    else:
        df.loc[(df[col] < -180) | (df[col] > 180), col] = np.nan

    # rounding values for consistency
    df[col] = df[col].round(6)

# checking null values after cleaning
for col in cols:
    print(col, "-> Nulls:", df[col].isnull().sum())

# preview cleaned data
df[cols].head()

GPS_Lat -> Nulls: 0
GPS_Lon -> Nulls: 0
Drop_Lat -> Nulls: 0
Drop_Lon -> Nulls: 0
Pickup_Lat -> Nulls: 0
Pickup_Lon -> Nulls: 0


,GPS_Lat,GPS_Lon,Drop_Lat,Drop_Lon,Pickup_Lat,Pickup_Lon
31,14.570000,75.198870,22.565754,88.379046,22.587249,88.358271
66,22.181200,86.392500,13.073789,80.282755,13.091661,80.271829
143,22.181200,86.392500,13.083241,80.264692,13.082735,80.265113
176,19.726761,81.268802,12.971982,77.579039,12.966111,77.589671
188,22.181200,86.392500,22.570784,88.345640,22.564980,88.369105


In [33]:
df.info()

<class 'pandas.DataFrame'>
Index: 7000 entries, 31 to 6976
Data columns (total 44 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Reservation_ID     7000 non-null   str           
 1   Customer_ID_x      7000 non-null   str           
 2   Vehicle_ID         7000 non-null   str           
 3   Vehicle_Class      7000 non-null   str           
 4   Booking_TS         7000 non-null   datetime64[us]
 5   Pickup_TS          7000 non-null   datetime64[us]
 6   Return_TS          7000 non-null   datetime64[us]
 7   City               7000 non-null   str           
 8   Pickup_Lat         7000 non-null   float64       
 9   Pickup_Lon         7000 non-null   float64       
 10  Drop_Lat           7000 non-null   float64       
 11  Drop_Lon           7000 non-null   float64       
 12  Odo_Start_km       7000 non-null   int64         
 13  Odo_End_km         7000 non-null   int64         
 14  Fuel_Fraction      7000

In [34]:
df.to_csv("car_rental_cleaned_dataset.csv", index=False)

In [35]:
df.info()

<class 'pandas.DataFrame'>
Index: 7000 entries, 31 to 6976
Data columns (total 44 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Reservation_ID     7000 non-null   str           
 1   Customer_ID_x      7000 non-null   str           
 2   Vehicle_ID         7000 non-null   str           
 3   Vehicle_Class      7000 non-null   str           
 4   Booking_TS         7000 non-null   datetime64[us]
 5   Pickup_TS          7000 non-null   datetime64[us]
 6   Return_TS          7000 non-null   datetime64[us]
 7   City               7000 non-null   str           
 8   Pickup_Lat         7000 non-null   float64       
 9   Pickup_Lon         7000 non-null   float64       
 10  Drop_Lat           7000 non-null   float64       
 11  Drop_Lon           7000 non-null   float64       
 12  Odo_Start_km       7000 non-null   int64         
 13  Odo_End_km         7000 non-null   int64         
 14  Fuel_Fraction      7000